<a href="https://www.kaggle.com/code/potalatarun/telugu-mlp-model?scriptVersionId=369971875" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# imports 
import pandas as pd
import re
import unicodedata
import torch
import torch.nn.functional as F

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
df = pd.read_csv("hf://datasets/Telugu-LLM-Labs/telugu_alpaca_yahma_cleaned_filtered_romanized/yahma_alpaca_cleaned_telugu_filtered_and_romanized.csv")

In [4]:


def is_telugu_word(word: str) -> bool:
    if not word:
        return False

    if not re.fullmatch(r"[\u0C00-\u0C7F]+", word):
        return False

    return any(
        unicodedata.category(ch).startswith("L")
        for ch in word
    )

In [5]:
# build telugu words dataset
words = []
for j, i in enumerate(df["telugu_instruction"]):
    words_ = i.split(" ")
    temp = []
    for word in words_:
        # if len(word) >= 6 and not word.isalpha() or not word.isalnum() or not word.isnumeric() or not word.isdigit():
        if len(word) >3 and is_telugu_word(word) == True:
            temp.append(word)
    words.extend(temp)
    # if j == 5:
    #     break
    

In [8]:
# below two cells import the names for the attached dataset

In [9]:
# with open("/kaggle/input/datasets/potalatarun/telugu-names-dataset/telugu_names_30000_candidates.txt", 'r') as file:
#     names = file.read().splitlines()

In [10]:
# len(names)

In [6]:
print(len(words))
words = list(set(words))
print(len(words))

177096
16532


In [13]:
vocab = sorted(list(set("".join(words))))
stoi = {s:i+1 for i, s in enumerate(vocab)}
stoi['.'] = 0
itos = {i:s for s, i in stoi.items()}

item_size = 3

xs, ys = [], []

for name in words:
    name += '.'
    context = [0] * item_size
    for n in name:
        n = stoi[n]
        xs.append(context)
        ys.append(n)
        context = context[1:] + [n]

X,y = torch.tensor(xs).to(device), torch.tensor(ys).to(device)

train_, valid_ = int(X.shape[0] * 80/100), int(X.shape[0] * 90 / 100)

X_train, y_train = X[:train_], y[:train_]
X_valid, y_valid = X[train_:valid_], y[train_:valid_]
X_test, y_test = X[valid_:], y[valid_:]

assert X_train.shape[0] == y_train.shape[0], "train dataset has different shapes for context and target"
assert X_valid.shape[0] == y_valid.shape[0], "valid dataset has different shapes for context and target"
assert X_test.shape[0] == y_test.shape[0], "test dataset has different shapes for context and target"


In [14]:
len(stoi)

62

In [15]:
# initialize the weights and embeddings layers dimentions
emb_dim = 100
h1_dim = 100
h2_dim = len(stoi)
vocab_len = len(stoi)

C = torch.randn((vocab_len, emb_dim)).to(device)
W1 = torch.randn((item_size * emb_dim, h1_dim)).to(device)
b1 = torch.randn((h1_dim)).to(device)
W2 = torch.randn((h1_dim, h2_dim)).to(device)
b2 = torch.randn((h2_dim)).to(device)

parameters = [C, W1, b1, W2, b2]

for p in parameters:
    p.requires_grad = True

In [23]:
# training loop
for i in range(10):
    embeds = C[X_train].view(-1, item_size*emb_dim)
    h1 = torch.tanh(embeds @ W1 + b1)
    logits = h1 @ W2 + b2

    loss = F.cross_entropy(logits, y_train)

    for p in parameters:
        p.grad = None

    loss.backward()

    for p in parameters:
        p.data += -0.1 * p.grad
    if i%100==0: print(i, "th epoch - loss: ", loss.item())
    

1.7102521657943726


In [29]:
# sample form the model

for _ in range(5):
    outputs = []
    inputs = [0, 0, 0]
    while True:
        embeds = C[inputs].view(-1, item_size * emb_dim)
        h1 = torch.tanh(embeds @ W1 + b1)
        logits = h1 @ W2 + b2
        probs = F.softmax(logits, dim=1)
        ix = torch.multinomial(probs, num_samples=1).item()
        output = itos[ix]
        # print(output)
        outputs.append(output)
        inputs = inputs[1:] + [ix]
        if ix==0:
            break
    print("".join(outputs))

స్కేటోని.
ప్రదిత.
టించి.
జాబితా.
రూపొంద.
వపనుక.
సెనుటిప్మిక్.
సరు.
గర్షాయఎ.
పిద్యాంత.
పదంశాలతోగౌగఢృత్రతిస్టమర్.
కలిగా.
బ్యాక్యంువుకవేటిుీరిష్కార్వీ.
రికు.
కోసం.
అర్థింగ్రిప్రొడక్ట్.
భావ్యబనన్య.
కోసం.
చేస్.
రెండు.
అనుస్తుడలుు.
వెబ్లో.
యొక్క.
ప్రిప్పుడ్దుగ్ైట్మేన్.
కంపోానికి.
భేజఞకఛం.
ఉద్దానికి.
చేయగలము.
ఎల్థం.
ప్రవర్గం.
ఎదురు.
ప్డి.
కంటే.
ఆన్.
టా.
భరోబల్.
చేయపాఠజఔఛభ.
కస్టమర్మింది.
అవసి.
ట్రశ్నకు.
ఉన్ని.
మరియు.
ప్రోపకానిశంలంలను.
శీకన.
సర్వలతమ్రోగ్రతిల.
చావశ్టర్య.
కలిగే.
ముడవ.
పాచుసఒను.
క్రెనో.


In [25]:
len(stoi)

62

In [26]:
words

['ఆరోగ్యంగా',
 'ఉండటానికి',
 'మూడు',
 'చిట్కాలు',
 'మూడు',
 'ప్రాధమిక',
 'రంగులు',
 'పరమాణువు',
 'నిర్మాణాన్ని',
 'వాయు',
 'కాలుష్యాన్ని',
 'మీరు',
 'స్ట్రక్షన్',
 'కంపెనీలో',
 'ప్రాజెక్ట్',
 'మేనేజర్',
 'మీరు',
 'క్లిష్టమైన',
 'నిర్ణయం',
 'తీసుకోవలసి',
 'వచ్చిన',
 'సమయాన్ని',
 'క్రింది',
 'వాటి',
 'సంక్షిప్త',
 'సారాంశాన్ని',
 'లేదా',
 'సిబిఎం',
 'ఆగస్టు',
 'కమోడోర్',
 'బిజినెస్',
 'మెషిన్',
 'డాలర్ల',
 'ప్రారంభ',
 'ధరతో',
 'తయారు',
 'హోమ్',
 'గణనీయమైన',
 'మార్కెట్',
 'విజయాన్ని',
 'అమ్మకాలు',
 'సుమారు',
 'మిలియన్',
 'యూనిట్లు',
 'టైమ్',
 'బెస్ట్',
 'సెల్లింగ్',
 'సింగిల్',
 'పర్సనల్',
 'కంప్యూటర్',
 'మోడల్',
 'కమోడోర్',
 'మరియు',
 'వాటా',
 'మరియు',
 'సంవత్సరానికి',
 'మిలియన్',
 'యూనిట్ల',
 'అమ్మకాలతో',
 'మార్కెట్లో',
 'ఆధిపత్యం',
 'ఐబిఎమ్',
 'పిసి',
 'ఆపిల్',
 'కంప్యూటర్లు',
 'మరియు',
 'అటారి',
 'కంప్యూటర్లను',
 'సామ్',
 'ట్రామియల్',
 'అటారి',
 'వారు',
 'కొన్ని',
 'సంవత్సరాలుగా',
 'నెలకు',
 'నిర్మిస్తున్నట్లు',
 'దిగువ',
 'భాగం',
 'ఎందుకు',
 'సమానమో',
 'క్యాంపింగ్',
 'ట్రిప్',
 'కోసం